# Tutorial 3 Exercise: SVM Classification for Hydraulic Fracturing

<h2>Table of Contents</h2>

<ol>
    <li><a href="#1">Problem Context &amp; Data Loading</a></li><br>
    <li><a href="#2">Feature Selection</a></li><br>
    <li><a href="#3">Data Preprocessing</a></li><br>
    <li><a href="#4">SVM Model &amp; Hyperparameter Tuning</a></li><br>
    <li><a href="#5">Model Evaluation</a></li><br>
    <li><a href="#6">Confusion Matrix Visualization</a></li><br>
    <li><a href="#7">Reflection</a></li>
</ol>

<hr id="1">
<h2>1. Problem Context & Data Loading</h2>

*Source: Hoss Belyadi, Alireza Haghighat, 2021. Machine Learning Guide for Oil and Gas Using Python*

**Problem Overview:**

In the Oil & Gas (O&G) industry, hydraulic fracturing (frac) treatments are crucial for stimulating well production. However, some stages within a frac can be unexpectedly challenging to treat, requiring additional chemicals and potentially leading to operational difficulties.

In the tutorial, we tackled this classification problem using Logistic Regression and KNN. Both models achieved reasonable accuracy, but the operations team has asked us to explore a more powerful approach. Support Vector Machines (SVM) can find complex, non-linear decision boundaries using the kernel trick, which may better capture the relationship between rock properties and frac difficulty.

**Objectives:**

Build, tune, and evaluate an SVM classifier that predicts whether a hydraulic frac stage will be challenging to treat (class 1) or not (class 0) based on wellbore measurements.

**Data:**

The dataset consists of historical frac stage data with the following features for each stage:

* **Measured Depth (MD_ft):** Depth of the stage within the wellbore (ft).
* **Resistivity:** Electrical resistance of the rock formation (ohm-m).
* **Young's Modulus / Poisson's Ratio (YM/PR):** Ratio of rock stiffness to deformation tendency (10^6 psi).
* **Gamma Ray (GR):** Natural radioactivity of the rock formation (gAPI).
* **Minimum Horizontal Stress Gradient:** Minimum stress acting on the formation in the horizontal direction (psi/ft).

The target variable is binary:

* **Class 0:** Stage was not challenging to frac.
* **Class 1:** Stage was challenging to frac, requiring additional treatment.

Import the necessary libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Read the data from `Fracability_DataSet.csv` into a Pandas DataFrame

In [ ]:
data = pd.read_csv('Fracability_DataSet.csv')
data.head()

In [ ]:
data.info()

In [ ]:
data.describe()

Let's visualize the class distribution to understand whether our dataset is balanced. An imbalanced dataset can bias the classifier toward the majority class.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
counts = data['Fracability'].value_counts().sort_index()
counts.plot(kind='bar', ax=ax, color=['steelblue', 'coral'])
ax.set_xlabel('Fracability Class')
ax.set_ylabel('Count')
ax.set_title('Class Distribution')
ax.set_xticklabels(['Not Challenging (0)', 'Challenging (1)'], rotation=0)

for i, v in enumerate(counts):
    ax.text(i, v + 5, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

<hr id="2">
<h2>2. Feature Selection</h2>

We now have a clear picture of the data. Before building our SVM model, we need to decide which features to include. Not all available measurements may be equally useful for predicting fracability, and including irrelevant features can hurt model performance.

Consider the physical meaning of each feature:
- **MD_ft** relates to the depth of the stage — deeper formations may behave differently under fracturing pressure.
- **Resistivity** reflects fluid content and porosity of the rock.
- **YM/PR** captures rock mechanical properties — stiffer rocks may fracture differently.
- **GR** indicates the lithology (rock type) from natural radioactivity.
- **Minimum Horizontal Stress Gradient** directly relates to the stress the rock is under, which is central to fracture initiation.

**Your task:** Choose the features you believe are most relevant for predicting whether a frac stage will be challenging. You can select anywhere from 2 to all 5 features. Think about which physical properties most directly influence fracture behavior.

In [ ]:
# Available features: 'MD_ft', 'Resistivity', 'YM/PR', 'GR', 'Minimum Horizontal Stress Gradient'

# TODO: Select the features you want to use for classification.
# Replace ___ with a list of feature column names.
# Example: selected_features = ['MD_ft', 'Resistivity']
selected_features = ___

x = data[selected_features]
y = data['Fracability']

print(f'Selected {len(selected_features)} features: {selected_features}')
print(f'Feature matrix shape: {x.shape}')

<hr id="3">
<h2>3. Data Preprocessing</h2>

With our features selected, we split the data into training and testing sets. SVM is sensitive to the scale of input features — a feature measured in thousands (like MD_ft) would dominate one measured in single digits (like YM/PR). We use `StandardScaler` to normalize all features to zero mean and unit variance before training.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Split the data
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# 2. Scale the features
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

print(f'Training set size: {x_train_scaled.shape[0]}')
print(f'Testing set size: {x_test_scaled.shape[0]}')

<hr id="4">
<h2>4. SVM Model & Hyperparameter Tuning</h2>

Now comes the core of the exercise. SVM performance depends heavily on the choice of hyperparameters:

- **C (Regularization):** Controls the trade-off between a smooth decision boundary and classifying training points correctly. Small C = smoother boundary (more regularization); large C = tighter fit to training data.
- **kernel:** Determines the type of decision boundary. `'rbf'` (radial basis function) can model non-linear boundaries; `'linear'` finds a straight hyperplane; `'sigmoid'` mimics a neural network activation.
- **gamma:** Controls how far the influence of a single training example reaches (only for `'rbf'` and `'sigmoid'` kernels). Small gamma = far reach (smoother); large gamma = close reach (more complex boundary).

We use `GridSearchCV` to systematically search across combinations of these parameters and find the best performing set via cross-validation.

Beyond hyperparameters, we also need to choose what metric to **optimize for** during the grid search. The `scoring` parameter in `GridSearchCV` determines which metric the search maximizes. Common choices include:

- `'accuracy'`: Overall fraction of correct predictions — treats all errors equally.
- `'precision'`: Of all stages predicted as *challenging*, how many actually were? Maximizing precision **reduces false positives** (false alarms).
- `'recall'`: Of all actually *challenging* stages, how many did we catch? Maximizing recall **reduces false negatives** (missed detections).
- `'f1'`: Harmonic mean of precision and recall — balances both types of error.

The operations team has reported that the current workflow over-prepares for too many stages that turn out to be straightforward, wasting chemicals and crew time. They want the model to **minimize these false alarms** (false positives) while still catching genuinely challenging stages.

**Your task:**
1. Define the hyperparameter search space — choose values for `C`, `kernel`, and `gamma`.
2. Choose  a suitable `scoring` method.

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

# 1. Create the SVM model
model = SVC()

# TODO: Define the hyperparameter grid for GridSearchCV.
# Choose values for C, kernel, and gamma to search over.
grid_params = {
    'C': ___,       # TODO: Provide a list of C values to try
    'kernel': ___,  # TODO: Provide a list of kernel types to try
    'gamma': ___,   # TODO: Provide a list of gamma values to try
}

# 2. Run GridSearchCV with 5-fold cross-validation
# TODO: Choose the scoring metric for GridSearchCV.
# Options: 'accuracy', 'precision', 'recall', 'f1'
# Think: which metric reduces the type of error the operations team cares about most?
grid_search = GridSearchCV(estimator=model, param_grid=grid_params, cv=5, scoring=___, n_jobs=-1)
grid_search.fit(x_train_scaled, y_train)

print('Grid search complete.')
print('Best Parameters:', grid_search.best_params_)
print(f'Best CV Score: {grid_search.best_score_:.4f}')

<hr id="5">
<h2>5. Model Evaluation</h2>

With the best hyperparameters identified, let's evaluate our tuned SVM on both the training and testing data. The classification report gives us precision, recall, and F1-score for each class.

In [ ]:
from sklearn.metrics import classification_report

# 1. Make predictions using the best model
y_pred_train = grid_search.predict(x_train_scaled)
y_pred_test = grid_search.predict(x_test_scaled)

# 2. Print classification reports
print('Training Data Classification Report:')
print(classification_report(y_train, y_pred_train))

print('Testing Data Classification Report:')
print(classification_report(y_test, y_pred_test))

<hr id="6">
<h2>6. Confusion Matrix Visualization</h2>

A classification report gives us aggregate metrics, but a confusion matrix lets us see exactly where the model gets it right and where it struggles. Each cell in the matrix shows the count of predictions:

- **Top-left (True Negatives):** Correctly predicted as *not challenging*.
- **Top-right (False Positives):** Predicted *challenging* but was actually *not challenging*.
- **Bottom-left (False Negatives):** Predicted *not challenging* but was actually *challenging*.
- **Bottom-right (True Positives):** Correctly predicted as *challenging*.

In [ ]:
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
labels = ['Not Challenging', 'Challenging']

# Training confusion matrix
cm_train = confusion_matrix(y_train, y_pred_train)
sns.heatmap(cm_train, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=labels, yticklabels=labels)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Training Set Confusion Matrix')

# Testing confusion matrix
cm_test = confusion_matrix(y_test, y_pred_test)
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=labels, yticklabels=labels)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].set_title('Testing Set Confusion Matrix')

plt.tight_layout()
plt.show()

<hr id="7">
<h2>7. Reflection</h2>

Answer the following questions based on your results:

**Question 1:** Which features did you select and why?

**Question 2:** What hyperparameter values did GridSearchCV identify as optimal? and Why?

**Question 3:** Compare the training and testing classification reports. Is the model generalizing well, or is there evidence of overfitting?

**Question 4:** Looking at the confusion matrix, which type of error (false positive vs false negative) is more common? Which would be more costly in a real frac operation?